# 04 Langzeit-Lichtkurve — Mehrere Nächte vergleichen

**Wann:** nach mindestens zwei abgeschlossenen NB02-Reihen desselben Ziels. **Ergebnis:** gemeinsame Lichtkurve aller Nächte, Phasenkurve und Nacht-für-Nacht-Übersicht.

| Feld | Ausfüllen || :--- | :--- || **Ziel** | z. B. `del Cep` || **Datenordner** | Ordner, unter dem alle Sitzungen liegen || **Periode** | bekannte Periode in Tagen (0 = keine Phasenfaltung) || **Epochen-Nullpunkt t₀** | BJD_TDB des Maximums (oder 0) |

## Formular

**1. Zelle unten ausführen (▶) — das Formular erscheint darunter. 2. Ausfüllen. 3. Nächste Zelle ausführen.**

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from varphot.hints import bind
from varphot.observer import data_dir as _data_dir

w_target = widgets.Text(description="Ziel", value="del Cep", layout=widgets.Layout(width="340px"))
w_datadir = widgets.Text(
    description="Datenordner",
    value=str(_data_dir()),
    layout=widgets.Layout(width="520px"),
)
w_period = widgets.FloatText(
    description="Periode [d]",
    value=0.0,
    step=0.0001,
    layout=widgets.Layout(width="240px"),
)
w_t0 = widgets.FloatText(
    description="t₀ [BJD_TDB]",
    value=0.0,
    step=0.1,
    layout=widgets.Layout(width="300px"),
)
w_binned = widgets.Checkbox(value=False, description="Tagesmittelwerte anzeigen")

bind(w_target, "nb04_target")
bind(w_datadir, "nb04_datadir")
bind(w_period, "nb04_period")
bind(w_t0, "nb04_t0")

display(
    widgets.VBox([
        widgets.HBox([w_target, w_period, w_t0]),
        w_datadir,
        w_binned,
    ])
)

## Ab hier automatisch

*Ausführen → Diese Zelle und alle folgenden.*

In [ ]:
from varphot.longterm import find_sessions, load_lightcurves, nightly_stats, phase_fold

TARGET  = w_target.value.strip()
DATADIR = w_datadir.value.strip()
PERIOD  = float(w_period.value)
T0      = float(w_t0.value)
BINNED  = bool(w_binned.value)

sessions = find_sessions(TARGET, DATADIR)
print(f"Gefundene Sitzungen für '{TARGET}': {len(sessions)}")
for s in sessions:
    print(f"  · {s}")
if not sessions:
    raise SystemExit(
        f"Keine abgeschlossenen Sitzungen für '{TARGET}' unter {DATADIR!r} gefunden.\n"
        "Bitte NB02 für mindestens eine Nacht ausführen und den richtigen Zielname eintragen."
    )

In [ ]:
import numpy as np
import pandas as pd

lc = load_lightcurves(sessions)
print(f"{len(lc)} Messpunkte, {lc['session'].nunique()} Nächte, "
      f"BJD_TDB {lc['bjd_tdb'].min():.4f} … {lc['bjd_tdb'].max():.4f}")
lc.head()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from astropy.time import Time

sessions_sorted = lc['session'].unique()
cmap = plt.cm.get_cmap('tab10', max(len(sessions_sorted), 1))
color_map = {s: cmap(i) for i, s in enumerate(sessions_sorted)}

nrows = 2 if PERIOD > 0 else 1
fig, axes = plt.subplots(nrows, 1, figsize=(12, 4 * nrows), constrained_layout=True)
if nrows == 1:
    axes = [axes]

# --- Rohkurve
ax0 = axes[0]
data_plot = nightly_stats(lc) if BINNED else lc
xcol = 'night_jd' if BINNED else 'bjd_tdb'
err_col = 'mag_err_mean' if BINNED else 'mag_err'
mag_col = 'mag_mean' if BINNED else 'mag'
for sess, grp in data_plot.groupby('session', sort=False):
    ax0.errorbar(
        grp[xcol], grp[mag_col], yerr=grp[err_col],
        fmt='o', ms=3, lw=0.6, capsize=2,
        color=color_map.get(sess, 'C0'), label=sess, alpha=0.8,
    )
ax0.invert_yaxis()
ax0.set_xlabel('BJD_TDB')
ax0.set_ylabel('mag')
ax0.set_title(f'{TARGET} — {len(sessions_sorted)} Nächte'
              + (' (Tagesmittelwerte)' if BINNED else ''))
ax0.legend(fontsize=7, ncol=min(4, len(sessions_sorted)))
ax0.grid(alpha=0.3)

# --- Phasenkurve (nur wenn Periode angegeben)
if PERIOD > 0:
    folded = phase_fold(lc, PERIOD, T0)
    ax1 = axes[1]
    for sess, grp in folded.groupby('session', sort=False):
        ax1.errorbar(
            grp['phase'], grp['mag'], yerr=grp['mag_err'],
            fmt='.', ms=3, lw=0.5, capsize=1.5,
            color=color_map.get(sess, 'C0'), label=sess, alpha=0.7,
        )
    ax1.invert_yaxis()
    ax1.set_xlabel(f'Phase (P = {PERIOD} d)')
    ax1.set_ylabel('mag')
    ax1.set_title(f'{TARGET} — Phasenkurve')
    ax1.legend(fontsize=7, ncol=min(4, len(sessions_sorted)))
    ax1.grid(alpha=0.3)

plt.savefig('langzeitkurve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik gespeichert: langzeitkurve.png')

In [ ]:
# Nacht-für-Nacht-Tabelle
nightly = nightly_stats(lc)
nightly['date_utc'] = [
    Time(bjd, format='jd', scale='tdb').to_datetime().strftime('%Y-%m-%d')
    for bjd in nightly['night_jd']
]
cols = ['date_utc', 'night_jd', 'n_points', 'mag_mean', 'mag_err_mean', 'mag_std', 'session']
nightly[cols].style \
    .format({'night_jd': '{:.4f}', 'mag_mean': '{:.4f}', 'mag_err_mean': '{:.4f}', 'mag_std': '{:.4f}'}) \
    .set_caption(f'Nächte — {TARGET}')